# MODULE 1 — Environment and Reproducibility

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

**Target platform:** MacBook Air M4 + Jupyter Notebook

**Purpose:** establish a reproducible software/hardware environment before touching either EEG dataset.

This module verifies:

- Python and operating-system information
- Apple Silicon / M4 detection
- PyTorch availability and MPS support
- optional TensorFlow availability
- scientific Python dependencies
- random seeds and deterministic settings where practical
- dataset/project paths
- experiment configuration
- device selection
- reproducibility metadata
- validation and leakage-policy configuration

**Important:** Module 1 does **not** load EEG data and does not fit any model.

## Scientific purpose

A reproducible cross-dataset EEG experiment must freeze the computational environment before data processing begins.

This notebook therefore creates a single configuration object that later modules will import/use. The configuration also explicitly records that the primary evaluation is **strict unseen-subject / calibration-free**.

The target-subject leakage rules are represented in code so later modules can reuse the same policy.

## Cell 1 — Python and package bootstrap

Run this cell first.

The installation logic is intentionally conservative:
- packages already installed are left alone;
- missing packages are installed with `%pip`;
- PyTorch is checked separately because Apple Silicon/MPS support is the primary deep-learning backend for this project.

In [1]:
# ============================================================
# CELL 1 — PACKAGE BOOTSTRAP
# ============================================================

from __future__ import annotations

import importlib
import platform
import subprocess
import sys

BASE_PACKAGES = {
    "numpy": "numpy>=1.26",
    "scipy": "scipy>=1.11",
    "pandas": "pandas>=2.1",
    "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
    "sklearn": "scikit-learn>=1.4",
    "mne": "mne>=1.6",
    "pyriemann": "pyriemann>=0.5",
    "einops": "einops>=0.7",
    "tqdm": "tqdm>=4.66",
    "psutil": "psutil>=5.9",
    "yaml": "pyyaml>=6.0",
    "umap": "umap-learn>=0.5",
}

# PyTorch is intentionally handled separately.
# On Apple Silicon, the official PyTorch package exposes MPS when
# the installed build and hardware/backend are compatible.
TORCH_PACKAGES = {
    "torch": "torch>=2.2",
}

def missing_packages(package_map):
    missing = []
    for module_name, pip_spec in package_map.items():
        try:
            importlib.import_module(module_name)
        except Exception:
            missing.append(pip_spec)
    return missing

missing_base = missing_packages(BASE_PACKAGES)
missing_torch = missing_packages(TORCH_PACKAGES)

if missing_base:
    print("Installing missing scientific packages:")
    for item in missing_base:
        print("  ", item)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_base])
else:
    print("All base scientific packages are already installed.")

if missing_torch:
    print("\nInstalling missing PyTorch package:")
    for item in missing_torch:
        print("  ", item)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_torch])
else:
    print("PyTorch is already installed.")

print("\nBootstrap complete.")
print("Python executable:", sys.executable)

Installing missing scientific packages:
   einops>=0.7
   umap-learn>=0.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 986.6 kB/s  0:00:02783.4 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 1.8 MB/s  0:00:22 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [umap-learn] 4/5 [umap-learn]
PyTorch is already installed.

Bootstrap complete.
Python executable: /opt/anaconda3/envs/mtech_tf/bin/python


## Cell 2 — Import libraries and inspect versions

This cell records the versions that will later be written into the experiment metadata.

Do not proceed if importing one of the required packages fails.

In [2]:
# ============================================================
# CELL 2 — IMPORTS + VERSION AUDIT
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import time
import random
import hashlib
import platform
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy
import matplotlib
import seaborn as sns
import sklearn
import mne
import pyriemann
import einops
import tqdm
import psutil
import yaml
import umap

import torch

try:
    import tensorflow as tf
    TENSORFLOW_AVAILABLE = True
except Exception as exc:
    tf = None
    TENSORFLOW_AVAILABLE = False
    TENSORFLOW_IMPORT_ERROR = repr(exc)

print("=" * 72)
print("MODULE 1 — PACKAGE VERSION AUDIT")
print("=" * 72)

versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "scikit_learn": sklearn.__version__,
    "mne": mne.__version__,
    "pyriemann": pyriemann.__version__,
    "einops": einops.__version__,
    "psutil": psutil.__version__,
    "umap_learn": umap.__version__,
    "pytorch": torch.__version__,
    "tensorflow": getattr(tf, "__version__", None),
}

for key, value in versions.items():
    print(f"{key:18s}: {value}")

if not TENSORFLOW_AVAILABLE:
    print("\nTensorFlow import status: NOT AVAILABLE")
    print("TensorFlow is optional for this project because the primary")
    print("training backend will be PyTorch + Apple MPS.")
    print("Import error:", TENSORFLOW_IMPORT_ERROR)

MODULE 1 — PACKAGE VERSION AUDIT
python            : 3.10.19
platform          : macOS-26.6.2-arm64-arm-64bit
machine           : arm64
processor         : arm
numpy             : 1.26.4
scipy             : 1.15.3
pandas            : 2.3.3
matplotlib        : 3.10.8
seaborn           : 0.13.2
scikit_learn      : 1.7.2
mne               : 1.11.0
pyriemann         : 0.10
einops            : 0.8.2
psutil            : 7.0.0
umap_learn        : 0.5.12
pytorch           : 2.10.0
tensorflow        : 2.16.2


## Cell 3 — Hardware and deep-learning backend checks

The experiment should automatically select:

1. **Apple MPS** when available;
2. otherwise CPU.

No CUDA-specific code is assumed.

The notebook also checks whether PyTorch sees an Apple GPU backend and whether a tiny tensor can actually execute on MPS.

In [3]:
# ============================================================
# CELL 3 — HARDWARE / MPS / CUDA CHECK
# ============================================================

print("=" * 72)
print("HARDWARE AND BACKEND CHECK")
print("=" * 72)

system = platform.system()
machine = platform.machine().lower()

print("Operating system :", system)
print("Architecture     :", machine)
print("CPU              :", platform.processor())
print("Logical CPUs     :", psutil.cpu_count(logical=True))
print("Physical CPUs    :", psutil.cpu_count(logical=False))
print("RAM (GB)         :", round(psutil.virtual_memory().total / (1024**3), 2))

is_apple_silicon = system == "Darwin" and machine in {"arm64", "aarch64"}
print("Apple Silicon    :", is_apple_silicon)

mps_built = hasattr(torch.backends, "mps") and torch.backends.mps.is_built()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
cuda_available = torch.cuda.is_available()

print("PyTorch MPS built:", mps_built)
print("PyTorch MPS avail:", mps_available)
print("CUDA available   :", cuda_available)

if mps_available:
    DEVICE = torch.device("mps")
elif cuda_available:
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Selected device  :", DEVICE)

# Minimal backend execution test
backend_test = {
    "device": str(DEVICE),
    "success": False,
    "error": None,
}

try:
    x = torch.randn(8, 8, device=DEVICE)
    y = x @ x.T
    backend_test["success"] = bool(torch.isfinite(y).all().item())
    print("Backend tensor test: PASS")
except Exception as exc:
    backend_test["error"] = repr(exc)
    print("Backend tensor test: FAIL")
    print("Error:", repr(exc))

if is_apple_silicon and not mps_available:
    print("\nWARNING:")
    print("Apple Silicon was detected, but PyTorch MPS is unavailable.")
    print("The project can still run on CPU, but deep-learning training")
    print("will be substantially slower until the PyTorch/MPS environment")
    print("is corrected.")

HARDWARE AND BACKEND CHECK
Operating system : Darwin
Architecture     : arm64
CPU              : arm
Logical CPUs     : 10
Physical CPUs    : 10
RAM (GB)         : 16.0
Apple Silicon    : True
PyTorch MPS built: True
PyTorch MPS avail: True
CUDA available   : False
Selected device  : mps
Backend tensor test: PASS


## Cell 4 — Reproducibility configuration

This cell defines the global seed and deterministic policy.

Important scientific caveat:

- deterministic behavior is requested where PyTorch supports it;
- Apple MPS can have operations that are not perfectly deterministic;
- exact bit-for-bit reproducibility is therefore not promised;
- all seeds, versions, device information and configuration values are saved.

In [4]:
# ============================================================
# CELL 4 — REPRODUCIBILITY CONFIGURATION
# ============================================================

GLOBAL_SEED = 20260822

def seed_everything(seed: int = GLOBAL_SEED) -> None:
    """
    Set reproducibility seeds where practical.

    Note:
    Exact bitwise determinism may not be guaranteed for every operation
    on Apple MPS. The seed and environment are nevertheless frozen and
    recorded for every experiment.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Ask PyTorch for deterministic algorithms where supported.
    # warn_only=True avoids crashing on a backend operation for which
    # deterministic behavior is unavailable.
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
        print("PyTorch deterministic algorithms: requested (warn_only=True)")
    except Exception as exc:
        print("Could not enable PyTorch deterministic algorithms:", repr(exc))

seed_everything()

# Additional backend/environment settings
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

print("Global seed:", GLOBAL_SEED)
print("PYTHONHASHSEED:", os.environ.get("PYTHONHASHSEED"))
print("Torch initial seed:", torch.initial_seed())

# Small repeatability sanity check on the selected device
seed_everything(GLOBAL_SEED)
test_a = torch.randn(16, device=DEVICE).detach().cpu().numpy()

seed_everything(GLOBAL_SEED)
test_b = torch.randn(16, device=DEVICE).detach().cpu().numpy()

repeatable = np.array_equal(test_a, test_b)

print("Seed repeatability check:", "PASS" if repeatable else "WARNING")

PyTorch deterministic algorithms: requested (warn_only=True)
Global seed: 20260822
PYTHONHASHSEED: 20260822
Torch initial seed: 20260822
PyTorch deterministic algorithms: requested (warn_only=True)
PyTorch deterministic algorithms: requested (warn_only=True)
Seed repeatability check: PASS


## Cell 5 — Project paths and experiment configuration

The local paths are those specified in the research brief.

This cell does **not** read any EEG files. It only checks whether the directories exist and records them.

If a path is missing, this is a configuration warning—not a reason to modify the path silently.

In [6]:
# ============================================================
# CELL 5 — PROJECT PATHS + CONFIGURATION
# ============================================================

PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")

BCI2A_ROOT = PROJECT_ROOT / "BCI IV-2a"
EEGMMIDB_ROOT = PROJECT_ROOT / "eegmmidb"

# Dedicated directories created by this project.
OUTPUT_ROOT = PROJECT_ROOT / "cross_dataset_mi_project"
CONFIG_ROOT = OUTPUT_ROOT / "config"
LOG_ROOT = OUTPUT_ROOT / "logs"
CACHE_ROOT = OUTPUT_ROOT / "cache"
RESULTS_ROOT = OUTPUT_ROOT / "results"
FIGURES_ROOT = OUTPUT_ROOT / "figures"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"

for path in [
    OUTPUT_ROOT,
    CONFIG_ROOT,
    LOG_ROOT,
    CACHE_ROOT,
    RESULTS_ROOT,
    FIGURES_ROOT,
    CHECKPOINT_ROOT,
    MANIFEST_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("PROJECT PATH AUDIT")
print("=" * 72)

print("Project root :", PROJECT_ROOT)
print("BCI-IV-2a    :", BCI2A_ROOT)
print("EEGMMIDB     :", EEGMMIDB_ROOT)
print("Output root  :", OUTPUT_ROOT)

print("\nDataset path existence:")
print("BCI-IV-2a    :", "PASS" if BCI2A_ROOT.exists() else "FAIL")
print("EEGMMIDB     :", "PASS" if EEGMMIDB_ROOT.exists() else "FAIL")

# Frozen project-level protocol from Module 0
EXPERIMENT_CONFIG = {
    "project_name": "cross_dataset_subject_independent_mi_eeg",
    "module": 1,
    "global_seed": GLOBAL_SEED,

    "paths": {
        "project_root": str(PROJECT_ROOT),
        "bci_iv_2a": str(BCI2A_ROOT),
        "eegmmidb": str(EEGMMIDB_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "config_root": str(CONFIG_ROOT),
        "log_root": str(LOG_ROOT),
        "cache_root": str(CACHE_ROOT),
        "results_root": str(RESULTS_ROOT),
        "figures_root": str(FIGURES_ROOT),
        "checkpoint_root": str(CHECKPOINT_ROOT),
        "manifest_root": str(MANIFEST_ROOT),
    },

    "task": {
        "name": "3-class motor imagery",
        "classes": ["left", "right", "feet"],
        "primary_dataset_1": "BCI-IV-2a",
        "primary_dataset_2": "EEGMMIDB",
        "unused_bci_iv_2a_class": "tongue",
    },

    "harmonization": {
        "target_sampling_rate_hz": 160,
        "primary_bandpass_hz": [8.0, 30.0],
        "secondary_bandpass_ablation_hz": [4.0, 40.0],
        "common_montage": "BCI-IV-2a common electrode space",
    },

    "evaluation": {
        "primary_protocol": "strict unseen-subject / calibration-free",
        "primary_split": "subject-level LOSO",
        "cross_dataset_directions": [
            "BCI-IV-2a -> EEGMMIDB",
            "EEGMMIDB -> BCI-IV-2a",
        ],
        "multi_source_domain_generalization": True,
        "target_data_allowed_during_training": False,
        "target_data_allowed_for_normalization": False,
        "target_data_allowed_for_feature_selection": False,
        "target_data_allowed_for_augmentation": False,
        "target_data_allowed_for_gan_training": False,
        "target_data_allowed_for_hyperparameter_tuning": False,
        "target_data_allowed_for_ssl_pretraining": False,
        "target_data_allowed_for_pseudo_labeling": False,
    },

    "model": {
        "primary_direction": "compact convolutional EEG encoder + Transformer/CCT",
        "initial_losses": [
            "cross_entropy",
            "domain_alignment",
            "supervised_cross_domain_contrastive",
        ],
        "main_model_framework": "PyTorch",
        "device_preference": ["mps", "cuda", "cpu"],
    },

    "reproducibility": {
        "seed": GLOBAL_SEED,
        "deterministic_algorithms_requested": True,
        "exact_bitwise_reproducibility_guaranteed": False,
        "note": (
            "Apple MPS may contain operations for which exact bitwise "
            "determinism is not guaranteed."
        ),
    },
}

config_json_path = CONFIG_ROOT / "module_1_experiment_config.json"

with open(config_json_path, "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)

print("\nConfiguration saved to:")
print(config_json_path)

PROJECT PATH AUDIT
Project root : /Users/ashokvarmabevara/Project2
BCI-IV-2a    : /Users/ashokvarmabevara/Project2/BCI IV-2a
EEGMMIDB     : /Users/ashokvarmabevara/Project2/eegmmidb
Output root  : /Users/ashokvarmabevara/Project2/cross_dataset_mi_project

Dataset path existence:
BCI-IV-2a    : PASS
EEGMMIDB     : PASS

Configuration saved to:
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/config/module_1_experiment_config.json


## Cell 6 — Environment metadata snapshot

The project should preserve the exact environment information used for an experiment.

This cell stores:
- package versions
- operating-system information
- hardware/backend information
- selected device
- timestamp
- project configuration

In [7]:
# ============================================================
# CELL 6 — ENVIRONMENT METADATA SNAPSHOT
# ============================================================

def get_git_commit(path: Path) -> Optional[str]:
    """
    Return the current Git commit if PROJECT_ROOT is a Git repository.
    Otherwise return None.
    """
    try:
        result = subprocess.run(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return None

ENVIRONMENT_METADATA = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "system": platform.system(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "logical_cpus": psutil.cpu_count(logical=True),
    "physical_cpus": psutil.cpu_count(logical=False),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 3),

    "torch_version": torch.__version__,
    "torch_mps_built": bool(mps_built),
    "torch_mps_available": bool(mps_available),
    "torch_cuda_available": bool(cuda_available),
    "selected_device": str(DEVICE),

    "tensorflow_available": bool(TENSORFLOW_AVAILABLE),
    "tensorflow_version": getattr(tf, "__version__", None),

    "package_versions": versions,
    "global_seed": GLOBAL_SEED,
    "backend_test": backend_test,

    "git_commit": get_git_commit(PROJECT_ROOT),
}

metadata_path = CONFIG_ROOT / "module_1_environment_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(ENVIRONMENT_METADATA, f, indent=2, default=str)

print("=" * 72)
print("ENVIRONMENT METADATA SAVED")
print("=" * 72)
print(metadata_path)

print("\nSelected backend:", DEVICE)
print("Git commit:", ENVIRONMENT_METADATA["git_commit"] or "No Git repository detected")

ENVIRONMENT METADATA SAVED
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/config/module_1_environment_metadata.json

Selected backend: mps
Git commit: No Git repository detected


## Cell 7 — Leakage policy lock

This cell creates a machine-readable policy that later modules can import.

The central rule is:

> The outer target subject is completely unavailable before final evaluation.

This policy is intentionally strict.

In [8]:
# ============================================================
# CELL 7 — LEAKAGE POLICY
# ============================================================

LEAKAGE_POLICY = {
    "strict_unseen_subject": True,

    "forbidden_target_information_before_final_test": [
        "training_trials",
        "validation_trials",
        "normalization_statistics",
        "reference_statistics",
        "CSP_filters",
        "FBCSP_filters",
        "feature_selection",
        "augmentation",
        "GAN_training",
        "hyperparameter_optimization",
        "early_stopping",
        "self_supervised_pretraining",
        "pseudo_labeling",
        "threshold_selection",
        "model_selection",
    ],

    "allowed_target_information_before_final_test": [],

    "required_checks_before_training": [
        "train_test_subject_disjoint",
        "train_test_trial_disjoint",
        "preprocessing_fit_only_on_training_data",
        "feature_fit_only_on_training_data",
        "augmentation_source_only",
        "hyperparameters_source_only",
        "SSL_source_only",
    ],

    "primary_claim": "strict_subject_independent_calibration_free",
}

leakage_path = CONFIG_ROOT / "strict_leakage_policy.json"

with open(leakage_path, "w", encoding="utf-8") as f:
    json.dump(LEAKAGE_POLICY, f, indent=2)

print("Leakage policy saved to:")
print(leakage_path)

print("\nPrimary claim:")
print(LEAKAGE_POLICY["primary_claim"])

print("\nTarget information allowed before final test:")
print(LEAKAGE_POLICY["allowed_target_information_before_final_test"])

Leakage policy saved to:
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/config/strict_leakage_policy.json

Primary claim:
strict_subject_independent_calibration_free

Target information allowed before final test:
[]


## Cell 8 — Module 1 validation report

This is the required validation gate.

The module passes only when:
- core imports succeed,
- an execution device is available,
- the backend tensor test succeeds,
- reproducibility seeding behaves as expected,
- the project configuration is saved,
- and the leakage policy explicitly forbids target-subject information before final testing.

A missing dataset directory is reported separately because dataset discovery belongs to Module 2.

In [9]:
# ============================================================
# CELL 8 — MODULE 1 VALIDATION REPORT
# ============================================================

validation = {}

validation["python_imports"] = True

validation["device_selected"] = DEVICE is not None
validation["backend_tensor_test"] = bool(backend_test["success"])
validation["seed_repeatability"] = bool(repeatable)

validation["config_saved"] = config_json_path.exists()
validation["metadata_saved"] = metadata_path.exists()
validation["leakage_policy_saved"] = leakage_path.exists()

validation["strict_target_isolation"] = (
    LEAKAGE_POLICY["strict_unseen_subject"]
    and LEAKAGE_POLICY["allowed_target_information_before_final_test"] == []
)

validation["dataset_paths_configured"] = (
    BCI2A_ROOT == Path("/Users/ashokvarmabevara/Project2/BCI IV-2a")
    and EEGMMIDB_ROOT == Path("/Users/ashokvarmabevara/Project2/eegmmib")
)

# TensorFlow is informational/optional in the project architecture.
validation["tensorflow_status"] = (
    "AVAILABLE" if TENSORFLOW_AVAILABLE else "OPTIONAL_NOT_AVAILABLE"
)

critical_checks = [
    validation["python_imports"],
    validation["device_selected"],
    validation["backend_tensor_test"],
    validation["seed_repeatability"],
    validation["config_saved"],
    validation["metadata_saved"],
    validation["leakage_policy_saved"],
    validation["strict_target_isolation"],
    validation["dataset_paths_configured"],
]

if all(critical_checks):
    module_status = "PASS"
else:
    # Dataset existence is not part of the Module 1 pass criterion.
    module_status = "FAIL"

print("\n" + "=" * 72)
print("MODULE VALIDATION REPORT — MODULE 1")
print("=" * 72)

for key, value in validation.items():
    print(f"{key:35s}: {value}")

print("\nDataset path status:")
print(f"  BCI-IV-2a exists : {BCI2A_ROOT.exists()}")
print(f"  EEGMMIDB exists  : {EEGMMIDB_ROOT.exists()}")

print("\nSTATUS:", module_status)

if module_status == "PASS":
    print("\nPASS")
    print("Environment, reproducibility configuration, device selection,")
    print("configuration persistence, and strict leakage policy are ready.")
    print("Proceed to Module 2 only after reviewing the environment outputs.")
else:
    print("\nFAIL")
    print("Do NOT proceed to Module 2 until the critical checks are fixed.")

if not mps_available and is_apple_silicon:
    print("\nWARNING: Apple Silicon detected but MPS is unavailable.")
    print("CPU execution is possible, but deep-learning training will be slower.")

if not BCI2A_ROOT.exists() or not EEGMMIDB_ROOT.exists():
    print("\nPASS WITH WARNING:")
    print("One or both dataset directories are not visible to this kernel.")
    print("Module 2 must resolve the actual local paths before loading data.")


MODULE VALIDATION REPORT — MODULE 1
python_imports                     : True
device_selected                    : True
backend_tensor_test                : True
seed_repeatability                 : True
config_saved                       : True
metadata_saved                     : True
leakage_policy_saved               : True
strict_target_isolation            : True
dataset_paths_configured           : False
tensorflow_status                  : AVAILABLE

Dataset path status:
  BCI-IV-2a exists : True
  EEGMMIDB exists  : True

STATUS: FAIL

FAIL
Do NOT proceed to Module 2 until the critical checks are fixed.


## Cell 9 — Human-readable run summary

Use this final cell as the record you can paste into experiment logs.

No EEG data are loaded in Module 1.

In [10]:
# ============================================================
# CELL 9 — HUMAN-READABLE RUN SUMMARY
# ============================================================

print("=" * 78)
print("MODULE 1 SUMMARY")
print("=" * 78)

print(f"Project                 : {EXPERIMENT_CONFIG['project_name']}")
print(f"Python                  : {versions['python']}")
print(f"OS                      : {platform.platform()}")
print(f"Architecture            : {platform.machine()}")
print(f"PyTorch                 : {torch.__version__}")
print(f"MPS built               : {mps_built}")
print(f"MPS available            : {mps_available}")
print(f"Selected device         : {DEVICE}")
print(f"Global seed             : {GLOBAL_SEED}")

print("\nPrimary experiment:")
print(f"  Task                  : {EXPERIMENT_CONFIG['task']['name']}")
print(f"  Classes               : {EXPERIMENT_CONFIG['task']['classes']}")
print(f"  Target sampling rate  : {EXPERIMENT_CONFIG['harmonization']['target_sampling_rate_hz']} Hz")
print(f"  Primary band          : {EXPERIMENT_CONFIG['harmonization']['primary_bandpass_hz']} Hz")
print(f"  Evaluation            : {EXPERIMENT_CONFIG['evaluation']['primary_protocol']}")
print(f"  LOSO                  : {EXPERIMENT_CONFIG['evaluation']['primary_split']}")

print("\nDataset paths:")
print(f"  BCI-IV-2a             : {BCI2A_ROOT}")
print(f"  EEGMMIDB              : {EEGMMIDB_ROOT}")

print("\nLeakage policy:")
print("  Target subject visible before final evaluation: NO")
print("  Target normalization statistics allowed      : NO")
print("  Target CSP/FBCSP fitting allowed             : NO")
print("  Target augmentation/GAN training allowed     : NO")
print("  Target hyperparameter tuning allowed         : NO")
print("  Target SSL pretraining allowed               : NO")

print("\nConfiguration files:")
print(f"  Experiment config     : {config_json_path}")
print(f"  Environment metadata  : {metadata_path}")
print(f"  Leakage policy        : {leakage_path}")

MODULE 1 SUMMARY
Project                 : cross_dataset_subject_independent_mi_eeg
Python                  : 3.10.19
OS                      : macOS-26.6.2-arm64-arm-64bit
Architecture            : arm64
PyTorch                 : 2.10.0
MPS built               : True
MPS available            : True
Selected device         : mps
Global seed             : 20260822

Primary experiment:
  Task                  : 3-class motor imagery
  Classes               : ['left', 'right', 'feet']
  Target sampling rate  : 160 Hz
  Primary band          : [8.0, 30.0] Hz
  Evaluation            : strict unseen-subject / calibration-free
  LOSO                  : subject-level LOSO

Dataset paths:
  BCI-IV-2a             : /Users/ashokvarmabevara/Project2/BCI IV-2a
  EEGMMIDB              : /Users/ashokvarmabevara/Project2/eegmmidb

Leakage policy:
  Target subject visible before final evaluation: NO
  Target normalization statistics allowed      : NO
  Target CSP/FBCSP fitting allowed             : NO
